In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler
import pickle

In [2]:
data=pd.read_csv("D:\GIT_CLONE\Kri_Na_ML_DL\ANN\ANN Classification\Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Preprocessing the data

In [3]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


## Encode Categorical Variable

In [4]:
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

In [5]:
# OHE for geography
ohe_geography=OneHotEncoder(handle_unknown='ignore')
ohe_geo=ohe_geography.fit_transform(data[['Geography']]).toarray()
ohe_geo_df=pd.DataFrame(ohe_geo,columns=ohe_geography.get_feature_names_out(['Geography']))
ohe_geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [6]:
data=pd.concat([data.drop('Geography',axis=1),ohe_geo_df],axis=1)
data.head()


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


## Splitting our data into independent and dependent

In [7]:
X=data.drop(['EstimatedSalary'],axis=1)
y=data['EstimatedSalary']
print(X.shape,y.shape)

(10000, 12) (10000,)


In [8]:
# Split the data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
# Standard Scaler
sscaler_x=StandardScaler()   
X_train=sscaler_x.fit_transform(X_train)
X_test=sscaler_x.transform(X_test)

In [10]:
# saving encoders into pickle
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('ohe_geography.pkl','wb') as file:
    pickle.dump(ohe_geography,file)

with open('sscaler_x.pkl','wb') as file:
    pickle.dump(sscaler_x,file)

In [11]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CreditScore        10000 non-null  int64  
 1   Gender             10000 non-null  int64  
 2   Age                10000 non-null  int64  
 3   Tenure             10000 non-null  int64  
 4   Balance            10000 non-null  float64
 5   NumOfProducts      10000 non-null  int64  
 6   HasCrCard          10000 non-null  int64  
 7   IsActiveMember     10000 non-null  int64  
 8   EstimatedSalary    10000 non-null  float64
 9   Exited             10000 non-null  int64  
 10  Geography_France   10000 non-null  float64
 11  Geography_Germany  10000 non-null  float64
 12  Geography_Spain    10000 non-null  float64
dtypes: float64(5), int64(8)
memory usage: 1015.8 KB


In [12]:
data.shape

(10000, 13)

## ANN Regression problem statement

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [14]:
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),  # Keras expects input_shape to be a tuple so we give for eg: (12,)
    Dense(32,activation='relu'),
    Dense(1) # here we are not giving activation because by default for activation fun for regression is linear regression
])

d:\GIT_CLONE\Kri_Na_ML_DL\ANN\ANN Classification\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
# Compile the model
model.compile(optimizer='Adam',loss=['mean_absolute_error'],metrics=['mae'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

# Setup tensorboard
log_dir="regressionlogs/fit/" + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
Tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)


In [17]:
# Setup Callback
earlystopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [18]:
# train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[Tensorboard_callback,earlystopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 100398.3828 - mae: 100398.3828 - val_loss: 98589.2891 - val_mae: 98589.2891
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 99859.6016 - mae: 99859.6016 - val_loss: 97457.3672 - val_mae: 97457.3672
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 97817.0703 - mae: 97817.0703 - val_loss: 94399.7266 - val_mae: 94399.7266
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 93647.6328 - mae: 93647.6328 - val_loss: 89109.3594 - val_mae: 89109.3594
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 87343.5469 - mae: 87343.5469 - val_loss: 81933.7812 - val_mae: 81933.7812
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 79462.5625 - mae: 79462.5625 - val_loss: 73754.8672 - val_mae: 73754.8672
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 71117.1094 - mae: 71117.1094 - val_loss: 65952.4141 - val_mae: 65952.4141
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step 

In [19]:
%load_ext tensorboard
%tensorboard --logdir regressionlogs/fit

Launching TensorBoard...

In [20]:
# Evaluate model on the test data
test_loss,test_mae=model.evaluate(X_test,y_test)
print(f"Test mae is {test_mae:.2f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 50260.3086 - mae: 50260.3086
Test mae is 50260.31


In [21]:
model.save('regressionmodel.h5')

In [22]:
# Streamlit is not created for regression problem